In [ ]:
import pandas as pd
import numpy as np
from pynm import pynm
import seaborn as sns
import matplotlib.pyplot as plt

import patsy as pat
import statsmodels.api as sm

## **Generate multisite data**
### Randomize metadata for each site cohort

In [ ]:
np.random.seed(666)
n_sites = 5
age_min = (np.random.rand(n_sites)*50).astype(int)
sites = pd.DataFrame(data={'sex_ratio': np.random.rand(n_sites),
                           'prob_ratio': 0.5*np.random.rand(n_sites),
                           'age_min': age_min,
                           'age_max': (age_min+5+np.random.rand(n_sites)*50).astype(int),
                           'score_shift': np.random.randn(n_sites)/4,
                           'sample_size': (50+np.random.rand(n_sites)*500).astype(int)})
sites

### Define implicit models

In [ ]:
def model(age,sex,offset):
    noise = np.random.normal(0,0.1)
    return 0.001*age-0.00001*(age-50)**2+0.5 + noise - np.random.uniform(0,0.3)*sex + offset

#Same model for probands, w/ more extreme noise and small random shift down
def model_prob(age,sex,offset):
    noise = np.random.normal(0,0.1)
    return 0.001*age-0.00001*(age-50)**2+0.5 + noise - np.random.uniform(0,0.3)*sex -0.2*np.random.uniform() + offset

### Generate fake participant data

In [ ]:
participants = []
for site in sites.iterrows():
    print(f'Processing site # {site[0]}')
    for participant in range(int(site[1]['sample_size'])):
        #sex = np.random.rand()>site[1]['sex_ratio']
        sex = np.random.binomial(1,site[1]['sex_ratio'])
        prob = np.random.binomial(1,site[1]['prob_ratio'])
        #age = site[1]['age_min'] + np.random.rand() * (site[1]['age_max']-site[1]['age_min'])
        age = np.random.uniform(site[1]['age_min'],site[1]['age_max'])
        if prob:
            score = model_prob(age,sex,site[1]['score_shift'])
        else:
            score = model(age,sex,site[1]['score_shift'])
        participants.append([site[0], sex,prob, age, score])

In [ ]:
df=pd.DataFrame(participants, columns=['site', 'sex','group', 'age', 'score'])
df.sex.replace({1: 'Female', 0: 'Male'}, inplace=True)
df.group.replace({1: 'PROB', 0: 'CTR'}, inplace=True)
df

In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(data=df, x='age', y='score', hue='site', style='site')
plt.show()

In [ ]:
%%time
import gpflow
from gpflow.utilities import print_summary
import tensorflow as tf

tf.random.set_seed(42)
np.random.set_seed(42)

length_scale=1
nu=2.5

k = gpflow.kernels.Matern12(variance=nu, lengthscales=length_scale) + gpflow.kernels.Constant() + gpflow.kernels.White(1)


ctr = df.loc[(df["group"] == "CTR")]
ctr_mask = df.index.isin(ctr.index)
probands = df.loc[(df["group"] == "PROB")]
prob_mask = df.index.isin(probands.index)

confounds, categorical = pynm.read_confounds(['age','C(sex)','C(site)'])
conf_mat = df[confounds]
conf_mat = pd.get_dummies(conf_mat,columns=categorical,drop_first=True)
conf_mat_cols = conf_mat.columns.tolist()
conf_mat = conf_mat.to_numpy()

y = df["score"][ctr_mask].to_numpy().reshape(-1,1)
X = conf_mat[ctr_mask]


m = gpflow.models.GPR(data=(X, y), kernel=k, mean_function=None)

opt = gpflow.optimizers.Scipy()
opt_logs = opt.minimize(m.training_loss, m.trainable_variables, options=dict(maxiter=1000))
print_summary(m)

In [ ]:
a = np.linspace(30, 80, 51).reshape(51, 1)
b = np.ones((51,5)) * 1
xx = np.hstack((a, b))

## predict mean and variance of latent GP at test points
mean, var = m.predict_f(xx)

## plot
plt.figure(figsize=(30, 20))
plt.plot(X[:, 0], y, "rx", mew=2)
plt.plot(xx[:, 0], mean, "C0", lw=2)
plt.fill_between(
    xx[:, 0],
    mean[:, 0] - 1.96 * np.sqrt(var[:, 0]),
    mean[:, 0] + 1.96 * np.sqrt(var[:, 0]),
    color="C0",
    alpha=0.2,
)

# ## generate 10 samples from posterior
#   # for reproducibility
# samples = m.predict_f_samples(X, 10)  # shape (10, 100, 1)
# plt.plot(X[:, 0], samples[:, :, 0].numpy().T, "bo", linewidth=0.5)

plt.show()

In [ ]:
i = np.argsort(X[:, 0])

## predict mean and variance of latent GP at test points
mean, var = m.predict_f(X[i, :])

## plot
plt.figure(figsize=(30, 20))
plt.plot(X[:, 0], y, "rx", mew=2)
plt.plot(X[i, 0], mean, "C0", lw=2)
plt.fill_between(
    X[i, 0],
    mean[:, 0] - 1.96 * np.sqrt(var[:, 0]),
    mean[:, 0] + 1.96 * np.sqrt(var[:, 0]),
    color="C0",
    alpha=0.2,
)

#plt.plot(X[:, 0], samples[:, :, 0].numpy().T, "bo", linewidth=0.5)
plt.show()

In [ ]:
%%time
tf.random.set_seed(42)
np.random.set_seed(42)

model = pynm.PyNM(df, 'score', 'group',
                  conf = 'age',
                  confounds = ['age', 'C(sex)', 'C(site)'])
gp = model.gp_normative_model()

In [ ]:
data_age_gp = model.data
data_age_gp.sort_values('age', inplace=True)

plt.figure(figsize=(30, 20))
plt.plot(X[:, 0], y, "rx", mew=2)
plt.plot(data_age_gp.age, data_age_gp.GP_nmodel_pred, "C0", lw=2)
plt.fill_between(
    np.squeeze(data_age_gp.age),
    np.squeeze(data_age_gp.GP_nmodel_pred) - 2*data_age_gp.GP_nmodel_sigma,
    np.squeeze(data_age_gp.GP_nmodel_pred) + 2*data_age_gp.GP_nmodel_sigma,
    color="C0",
    alpha=0.2,
)
plt.show()